# Analytics - Mapa da Cidadania e Acesso a Informacao DF

Notebook exploratorio para consumir os 8 arquivos CSV/XLSX da camada raw e gerar uma primeira leitura analitica sobre LAI, PDAD e projecoes populacionais.

## 1. Importacoes e Configuracoes

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('Set2')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 11

def localizar_raw_dir() -> Path:
    cwd = Path.cwd()
    candidatos = [
        cwd,
        cwd / 'Data Layer' / 'raw',
        cwd / 'Data Layer' / '01_raw',
        cwd.parent / 'raw',
        cwd.parent / '01_raw',
    ]
    for candidato in candidatos:
        if (candidato / 'analyticsExemplo.ipynb').exists() or (candidato / 'moradores.csv').exists():
            return candidato
    raise FileNotFoundError('Nao foi possivel localizar a pasta raw com os arquivos de entrada.')

RAW_DIR = localizar_raw_dir()
RAW_DIR

## 2. Carregamento dos Dados

### Contexto de negocio

A base combina tres familias de informacao:
- **Participa DF / LAI**: pedidos, recursos, pesquisa de satisfacao e data de atualizacao.
- **PDAD**: moradores, domicilios e dicionario de variaveis.
- **Projecoes populacionais**: populacao por Regiao Administrativa, sexo e faixa etaria entre 2020 e 2030.

O objetivo desta camada analitica e validar cobertura, qualidade e sinais iniciais para alimentar as camadas silver e gold da arquitetura medalhao.

In [ ]:
ARQUIVOS = {
    'moradores': 'moradores.csv',
    'domicilios': 'domicilios.xlsx',
    'dicionario': 'dicionario_de_variaveis_pdada_2024_público.xlsx',
    'lai_pedidos': 'Participa DF _LAI_ - Pedidos.csv',
    'lai_recursos': 'Participa DF _LAI_ - Recursos.csv',
    'lai_satisfacao': 'Participa DF _LAI_ - Pesquisa de Satisfação.csv',
    'lai_atualizacao': 'Participa DF _LAI_ - Última Atualização.csv',
    'projecoes_populacionais': 'Dados - Projeções populacionais por Região Administrativa, sexo e faixa etária 2020-2030_1.xlsx',
}

for nome, arquivo in ARQUIVOS.items():
    caminho = RAW_DIR / arquivo
    print(f'{nome:24} {"OK" if caminho.exists() else "NAO ENCONTRADO"} - {caminho.name}')

In [ ]:
def ler_csv_raw(nome_arquivo: str, **kwargs) -> pd.DataFrame:
    caminho = RAW_DIR / nome_arquivo
    tentativas = ['utf-8-sig', 'cp1252', 'latin1']
    ultimo_erro = None
    for encoding in tentativas:
        try:
            return pd.read_csv(caminho, sep=';', encoding=encoding, low_memory=False, **kwargs)
        except UnicodeDecodeError as erro:
            ultimo_erro = erro
    raise ultimo_erro

def ler_excel_raw(nome_arquivo: str, **kwargs):
    return pd.read_excel(RAW_DIR / nome_arquivo, engine='openpyxl', **kwargs)

df_moradores = ler_csv_raw(ARQUIVOS['moradores'])
df_domicilios = ler_excel_raw(ARQUIVOS['domicilios'])
df_lai_pedidos = ler_csv_raw(ARQUIVOS['lai_pedidos'])
df_lai_recursos = ler_csv_raw(ARQUIVOS['lai_recursos'])
df_lai_satisfacao = ler_csv_raw(ARQUIVOS['lai_satisfacao'])
df_lai_atualizacao = ler_csv_raw(ARQUIVOS['lai_atualizacao'])

dict_dicionario = ler_excel_raw(ARQUIVOS['dicionario'], sheet_name=None)
dict_projecoes = ler_excel_raw(ARQUIVOS['projecoes_populacionais'], sheet_name=None)

datasets = {
    'moradores': df_moradores,
    'domicilios': df_domicilios,
    'lai_pedidos': df_lai_pedidos,
    'lai_recursos': df_lai_recursos,
    'lai_satisfacao': df_lai_satisfacao,
    'lai_atualizacao': df_lai_atualizacao,
}

resumo_carga = pd.DataFrame([
    {'dataset': nome, 'linhas': len(df), 'colunas': df.shape[1]}
    for nome, df in datasets.items()
])

resumo_carga.loc[len(resumo_carga)] = ['dicionario_abas', len(dict_dicionario), np.nan]
resumo_carga.loc[len(resumo_carga)] = ['projecoes_abas', len(dict_projecoes), np.nan]
resumo_carga

## 3. Qualidade dos Dados

### Insight de negocio

Antes da transformacao para a camada silver, a prioridade e identificar bases com alta incompletude, colunas duplicadas semanticamente e campos de data que precisam de padronizacao. Isso reduz risco de metricas inconsistentes na camada gold.

In [ ]:
qualidade = []
for nome, df in datasets.items():
    total_celulas = df.shape[0] * df.shape[1]
    nulos = int(df.isna().sum().sum())
    qualidade.append({
        'dataset': nome,
        'linhas': df.shape[0],
        'colunas': df.shape[1],
        'celulas_nulas': nulos,
        'pct_nulos': round((nulos / total_celulas) * 100, 2) if total_celulas else 0,
        'linhas_duplicadas': int(df.duplicated().sum()),
    })

df_qualidade = pd.DataFrame(qualidade).sort_values('pct_nulos', ascending=False)
df_qualidade

In [ ]:
def top_nulos(df: pd.DataFrame, n: int = 15) -> pd.DataFrame:
    nulos = df.isna().sum()
    pct = (nulos / len(df) * 100).round(2)
    return (
        pd.DataFrame({'coluna': nulos.index, 'nulos': nulos.values, 'pct_nulos': pct.values})
        .query('nulos > 0')
        .sort_values('pct_nulos', ascending=False)
        .head(n)
    )

top_nulos(df_lai_pedidos)

## 4. LAI: Pedidos de Acesso a Informacao

### 4.1 Evolucao temporal dos pedidos

**Insight de negocio:** a evolucao anual indica a demanda social por informacao publica e ajuda a dimensionar capacidade operacional dos orgaos que respondem pedidos de LAI.

In [ ]:
df_lai_pedidos['DT_PEDIDO_PARSE'] = pd.to_datetime(df_lai_pedidos['DT_PEDIDO'], dayfirst=True, errors='coerce')
df_lai_pedidos['ANO_PEDIDO'] = df_lai_pedidos['DT_PEDIDO_PARSE'].dt.year.fillna(df_lai_pedidos.get('NR_ANO_PEDIDO')).astype('Int64')

pedidos_por_ano = df_lai_pedidos['ANO_PEDIDO'].value_counts().sort_index()

plt.figure(figsize=(14, 6))
ax = sns.barplot(x=pedidos_por_ano.index.astype(str), y=pedidos_por_ano.values, color='#2c7fb8')
ax.set_title('Evolucao anual dos pedidos LAI', fontsize=14, fontweight='bold')
ax.set_xlabel('Ano')
ax.set_ylabel('Quantidade de pedidos')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print(f'Total de pedidos: {len(df_lai_pedidos):,}')
print(f'Ano com mais pedidos: {pedidos_por_ano.idxmax()} ({pedidos_por_ano.max():,})')

### 4.2 Situacao, prazo e classificacao das respostas

**Insight de negocio:** situacao e prazo mostram maturidade do processo de atendimento. Classificacoes de resposta ajudam a separar acesso concedido, negativas e casos que exigem tratamento adicional.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

situacao = df_lai_pedidos['DS_SITUACAO_PEDIDO'].fillna('Nao informado').value_counts().head(10)
sns.barplot(y=situacao.index, x=situacao.values, ax=axes[0], color='#41ab5d')
axes[0].set_title('Top situacoes dos pedidos')
axes[0].set_xlabel('Pedidos')
axes[0].set_ylabel('')

prazo = df_lai_pedidos['STATUS_PRAZO_RESPOSTA'].fillna('Nao informado').value_counts().head(10)
sns.barplot(y=prazo.index, x=prazo.values, ax=axes[1], color='#fdae61')
axes[1].set_title('Status do prazo de resposta')
axes[1].set_xlabel('Pedidos')
axes[1].set_ylabel('')

plt.tight_layout()
plt.show()

classificacao = df_lai_pedidos['RESPOSTAS.DS_TIPO_CLASSIFICACAO_RESPOSTA'].fillna('Nao informado').value_counts().head(12)
classificacao.to_frame('pedidos')

### 4.3 Orgaos mais demandados

**Insight de negocio:** orgaos com maior volume de pedidos tendem a concentrar temas de maior interesse publico e devem ser priorizados em paineis de transparencia ativa.

In [ ]:
orgaos = df_lai_pedidos['DS_UNIDADE'].fillna('Nao informado').value_counts().head(15)

plt.figure(figsize=(14, 8))
sns.barplot(y=orgaos.index, x=orgaos.values, color='#756bb1')
plt.title('Top 15 orgaos por volume de pedidos LAI', fontsize=14, fontweight='bold')
plt.xlabel('Pedidos')
plt.ylabel('Orgao')
plt.tight_layout()
plt.show()

orgaos.to_frame('pedidos')

## 5. LAI: Recursos e Pesquisa de Satisfacao

### Insight de negocio

Recursos indicam friccao no atendimento original. Satisfacao mostra a percepcao do usuario e pode orientar melhorias de linguagem, completude e tempestividade das respostas.

In [ ]:
df_lai_recursos['DT_RECURSO_PARSE'] = pd.to_datetime(df_lai_recursos['DT_RECURSO'], dayfirst=True, errors='coerce')
df_lai_recursos['ANO_RECURSO'] = df_lai_recursos['DT_RECURSO_PARSE'].dt.year

recursos_por_ano = df_lai_recursos['ANO_RECURSO'].value_counts().sort_index()
instancias = df_lai_recursos['DS_INSTANCIA'].fillna('Nao informado').value_counts()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.barplot(x=recursos_por_ano.index.astype(str), y=recursos_por_ano.values, ax=axes[0], color='#d95f0e')
axes[0].set_title('Recursos por ano')
axes[0].set_xlabel('Ano')
axes[0].set_ylabel('Recursos')
axes[0].tick_params(axis='x', rotation=45)

sns.barplot(y=instancias.index, x=instancias.values, ax=axes[1], color='#636363')
axes[1].set_title('Recursos por instancia')
axes[1].set_xlabel('Recursos')
axes[1].set_ylabel('')

plt.tight_layout()
plt.show()

print(f'Total de recursos: {len(df_lai_recursos):,}')
print(f'Pedidos com recurso: {df_lai_recursos["NR_PEDIDO_PARTICIPA"].nunique():,}')

In [ ]:
questoes = df_lai_satisfacao['DS_QUESTAO'].fillna('Nao informado').value_counts().head(10)
alternativas = df_lai_satisfacao['DS_ALTERNATIVA'].fillna('Nao informado').value_counts().head(10)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.barplot(y=questoes.index, x=questoes.values, ax=axes[0], color='#1b9e77')
axes[0].set_title('Questoes mais respondidas')
axes[0].set_xlabel('Respostas')
axes[0].set_ylabel('')

sns.barplot(y=alternativas.index, x=alternativas.values, ax=axes[1], color='#e7298a')
axes[1].set_title('Alternativas de satisfacao')
axes[1].set_xlabel('Respostas')
axes[1].set_ylabel('')

plt.tight_layout()
plt.show()

df_lai_atualizacao

## 6. PDAD: Moradores e Domicilios

### Insight de negocio

A PDAD sustenta o perfil socioeconomico por Regiao Administrativa. Nesta etapa exploratoria, o foco e conferir cobertura, pesos amostrais e variaveis de renda, idade, escolaridade e domicilio para orientar a modelagem silver.

In [ ]:
colunas_chave_moradores = [c for c in ['localidade', 'idade_calculada', 'renda_ind', 'renda_ind_r', 'escolaridade', 'peso_mor'] if c in df_moradores.columns]
colunas_chave_domicilios = [c for c in ['localidade', 'renda_dom', 'renda_dom_r', 'peso_dom'] if c in df_domicilios.columns]

print('Colunas-chave moradores:', colunas_chave_moradores)
print('Colunas-chave domicilios:', colunas_chave_domicilios)

display(df_moradores[colunas_chave_moradores].head())
display(df_domicilios[colunas_chave_domicilios].head() if colunas_chave_domicilios else df_domicilios.head())

In [ ]:
if 'idade_calculada' in df_moradores.columns:
    idade = pd.to_numeric(df_moradores['idade_calculada'], errors='coerce')
    idade = idade[(idade >= 0) & (idade <= 120)]

    plt.figure(figsize=(14, 6))
    sns.histplot(idade, bins=30, kde=True, color='#3182bd')
    plt.title('Distribuicao de idade dos moradores', fontsize=14, fontweight='bold')
    plt.xlabel('Idade')
    plt.ylabel('Moradores')
    plt.tight_layout()
    plt.show()

    print(f'Idade media: {idade.mean():.1f}')
    print(f'Idade mediana: {idade.median():.1f}')

In [ ]:
if {'localidade', 'peso_mor'}.issubset(df_moradores.columns):
    df_moradores['peso_mor_num'] = pd.to_numeric(
        df_moradores['peso_mor'].astype(str).str.replace(',', '.', regex=False),
        errors='coerce'
    )
    moradores_por_ra = (
        df_moradores.groupby('localidade', dropna=False)['peso_mor_num']
        .sum()
        .sort_values(ascending=False)
        .head(20)
    )

    plt.figure(figsize=(14, 8))
    sns.barplot(y=moradores_por_ra.index.astype(str), x=moradores_por_ra.values, color='#31a354')
    plt.title('Top 20 localidades por soma do peso amostral de moradores', fontsize=14, fontweight='bold')
    plt.xlabel('Soma do peso')
    plt.ylabel('Localidade')
    plt.tight_layout()
    plt.show()

    moradores_por_ra.to_frame('peso_moradores')

## 7. Projecoes Populacionais

### Insight de negocio

As projecoes populacionais permitem normalizar indicadores por porte populacional e comparar regioes de tamanhos diferentes. Isso e essencial para criar indicadores como pedidos LAI por 10 mil habitantes.

In [ ]:
abas_projecoes = pd.DataFrame({
    'aba': list(dict_projecoes.keys()),
    'linhas': [len(df) for df in dict_projecoes.values()],
    'colunas': [df.shape[1] for df in dict_projecoes.values()],
})
abas_projecoes

In [ ]:
projecoes_longas = []
for aba, df in dict_projecoes.items():
    temp = df.copy()
    temp['regiao_administrativa_aba'] = aba
    projecoes_longas.append(temp)

df_projecoes = pd.concat(projecoes_longas, ignore_index=True)

colunas_ano = [c for c in df_projecoes.columns if str(c).isdigit() and 2020 <= int(c) <= 2030]
print('Colunas de ano identificadas:', colunas_ano)

if colunas_ano:
    proj_total_ano = df_projecoes[colunas_ano].apply(pd.to_numeric, errors='coerce').sum().sort_index()
    plt.figure(figsize=(14, 6))
    sns.lineplot(x=proj_total_ano.index.astype(str), y=proj_total_ano.values, marker='o', color='#08519c')
    plt.title('Tendencia agregada das projecoes populacionais', fontsize=14, fontweight='bold')
    plt.xlabel('Ano')
    plt.ylabel('Populacao projetada')
    plt.tight_layout()
    plt.show()

df_projecoes.head()

## 8. Dicionario de Dados

### Insight de negocio

O dicionario deve ser usado como contrato semantico da camada silver. Ele ajuda a traduzir codigos da PDAD para categorias legiveis e reduz ambiguidade na criacao de dimensoes e fatos.

In [ ]:
abas_dicionario = pd.DataFrame({
    'aba': list(dict_dicionario.keys()),
    'linhas': [len(df) for df in dict_dicionario.values()],
    'colunas': [df.shape[1] for df in dict_dicionario.values()],
})
abas_dicionario

In [ ]:
for aba in ['moradores', 'domicilios']:
    if aba in dict_dicionario:
        print(f'\n=== {aba} ===')
        display(dict_dicionario[aba].head(10))

## 9. Sinais para a Camada Silver e Gold

### Recomendacoes tecnicas

1. Padronizar nomes de colunas em `snake_case` na camada silver.
2. Converter datas LAI para `datetime` e derivar ano, mes e tempo de resposta.
3. Tratar codigos sentinela da PDAD, como `99999` e `88888`, como valores especiais antes de calcular indicadores.
4. Usar o dicionario da PDAD para traduzir categorias codificadas.
5. Transformar projecoes populacionais para formato longo: `regiao_administrativa`, `sexo`, `faixa_etaria`, `ano`, `populacao`.
6. Na gold, priorizar fatos de pedidos LAI, perfil socioeconomico e populacao, conectados por dimensoes de tempo e Regiao Administrativa.